# Lab 08: Monitoring & Insights

## Business Context

The analyzer is live. Now you need to know when it breaks. Enable monitoring, detect drift, and close the feedback loop. Final analysis: is there a pattern between how clearly a company explains itself and how the stock performs?

**After this lab:** The endpoint has inference table logging. Lakehouse Monitor runs automated drift detection. And the correlation question the firm's analysts wanted answered — clarity vs. stock returns — finally has data behind it.

| Attribute | Detail |
|---|---|
| **Key Skills** | `auto_capture_config`, inference tables, Lakehouse Monitor, drift detection, feedback loop |
| **Estimated Time** | 30 minutes |
| **Estimated Cost** | ~$2-3 |

In [ ]:
%pip install databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
import json
import time
import requests
from databricks.sdk import WorkspaceClient

CATALOG       = "ipo_analyzer"
SCHEMA        = "default"
LLM_ENDPOINT  = "databricks-meta-llama-3.1-405b-instruct"
ENDPOINT_NAME = "ipo-analyzer-endpoint"

print(f"Catalog        : {CATALOG}")
print(f"Schema         : {SCHEMA}")
print(f"LLM            : {LLM_ENDPOINT}")
print(f"Endpoint name  : {ENDPOINT_NAME}")

w = WorkspaceClient()
print(f"WorkspaceClient: ready (host={w.config.host})")

## A. Enable Inference Tables

**Inference tables** (also called payload logging) automatically record every request and response that passes through a serving endpoint into a Delta table. This is the foundation of production monitoring: you can't monitor what you don't log.

Configuration is done via the serving endpoint REST API using **`auto_capture_config`**:

| Field | Value | Meaning |
|---|---|---|
| `catalog_name` | `ipo_analyzer` | Unity Catalog catalog where the log table is stored |
| `schema_name` | `default` | Schema within the catalog |
| `table_name_prefix` | `ipo_endpoint` | Prefix for the generated table name |
| `enabled` | `True` | Activate logging immediately |

Databricks creates the table automatically as `{catalog}.{schema}.{prefix}_{endpoint_name}`. Every request/response pair lands there within seconds of being served.

We use `requests.patch` against the REST API because the SDK's `update_config_and_wait` is for model config changes; `auto_capture_config` is a separate endpoint-level setting.

In [ ]:
try:
    # Enable auto_capture_config via REST API
    host  = w.config.host
    token = w.config.token
    
    url = f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config"
    
    payload = {
        "auto_capture_config": {
            "catalog_name": CATALOG,
            "schema_name": SCHEMA,
            "table_name_prefix": "ipo_endpoint",
            "enabled": True,
        }
    }
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }
    
    response = requests.patch(url, headers=headers, json=payload)
    response.raise_for_status()
    
    print("=== auto_capture_config enabled ===")
    result = response.json()
    capture_cfg = result.get("auto_capture_config", {})
    print(f"  Catalog      : {capture_cfg.get('catalog_name')}")
    print(f"  Schema       : {capture_cfg.get('schema_name')}")
    print(f"  Table prefix : {capture_cfg.get('table_name_prefix')}")
    print(f"  Enabled      : {capture_cfg.get('enabled')}")
    print()
    print("Inference table will be created at:")
    print(f"  {CATALOG}.{SCHEMA}.ipo_endpoint_payload")
except Exception as e:
    print(f'Skipped (endpoint not deployed): {str(e)[:200]}')


## B. Generate Traffic

Monitoring needs data. We send 8 representative queries to the endpoint — covering the core use cases the agent was built for. With inference tables enabled, every request/response pair is logged automatically.

We pause 2 seconds between requests to produce a natural traffic pattern (rather than a burst), which gives Lakehouse Monitor more realistic data to analyze.

In [ ]:
try:
    # 8 representative queries covering core use cases
    queries = [
        "What are Snowflake's key risk factors according to their S-1?",
        "How does Coinbase generate revenue?",
        "What market opportunity did DoorDash claim in their S-1 filing?",
        "Summarise Palantir's government contract strategy.",
        "What manufacturing risks did Rivian disclose?",
        "Which IPO companies had the highest 12-month returns?",
        "Show me the top 5 performers with their clarity scores.",
        "Which company had the clearest S-1 filing based on clarity scores?",
    ]
    
    print(f"Sending {len(queries)} queries to '{ENDPOINT_NAME}'...\n")
    
    for i, query in enumerate(queries, 1):
        try:
            resp = w.serving_endpoints.query(
                name=ENDPOINT_NAME,
                dataframe_records=[{"input": query}],
            )
            answer = resp.as_dict().get("predictions", [{}])[0]
            preview = str(answer)[:80].replace("\n", " ")
            print(f"[{i}/{len(queries)}] Query sent. Response preview: {preview}...")
        except Exception as e:
            print(f"[{i}/{len(queries)}] Error: {e}")
    
        if i < len(queries):
            time.sleep(2)
    
    print("\nAll queries sent. Inference table will populate within ~60 seconds.")
except Exception as e:
    print(f'Skipped (endpoint not deployed): {str(e)[:200]}')


In [ ]:
try:
    # Inspect the inference table
    # Table name follows the pattern: {prefix}_{endpoint_name} with hyphens replaced by underscores
    inference_table = f"{CATALOG}.{SCHEMA}.ipo_endpoint_payload"
    
    print(f"Inference table: {inference_table}")
    print()
    
    try:
        display(spark.sql(f"""
            SELECT
                timestamp,
                request,
                response,
                status_code,
                execution_time_ms
            FROM {inference_table}
            ORDER BY timestamp DESC
            LIMIT 10
        """))
    except Exception as e:
        print(f"Note: Inference table may not be visible yet — it can take 1-2 minutes to appear.")
        print(f"Re-run this cell after waiting. Error: {e}")
except Exception as e:
    print(f'Skipped (endpoint not deployed): {str(e)[:200]}')


## C. Create Lakehouse Monitor

**Lakehouse Monitor** provides automated data quality and drift detection over any Delta table. For inference tables, it tracks:

- **Data drift** — are incoming requests changing over time? (detected via Jensen-Shannon divergence on token distributions)
- **Prediction drift** — are response patterns changing? (length, structure, refusal rate)
- **Data quality** — are there nulls, schema changes, unexpected values?

`MonitorInferenceLog` is the profile type for ML model inference data. Key parameters:

| Parameter | Value | Meaning |
|---|---|---|
| `granularities` | `["1 day"]` | Compute drift statistics daily |
| `timestamp_col` | `"timestamp"` | Column used to bucket rows into time windows |
| `prediction_col` | `"response"` | Column containing model output (what we monitor for drift) |
| `model_id_col` | (optional) | Column identifying which model version served each request |

`run_refresh()` triggers the first computation immediately — otherwise the monitor runs on its configured schedule.

In [ ]:
try:
    from databricks.sdk.service.catalog import MonitorInferenceLog
    
    inference_table = f"{CATALOG}.{SCHEMA}.ipo_endpoint_payload"
    
    try:
        monitor = w.quality_monitors.create(
            table_name=inference_table,
            inference_log=MonitorInferenceLog(
                granularities=["1 day"],
                timestamp_col="timestamp",
                prediction_col="response",
            ),
            output_schema_name=f"{CATALOG}.{SCHEMA}",
        )
        print(f"Monitor created for: {inference_table}")
        print(f"Status : {monitor.status}")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"Monitor already exists for '{inference_table}' — skipping creation.")
            monitor = w.quality_monitors.get(table_name=inference_table)
            print(f"Status : {monitor.status}")
        else:
            print(f"Monitor creation note: {e}")
            print("This is expected if the inference table does not yet exist.")
            print("Re-run after waiting ~2 minutes for the inference table to populate.")
except Exception as e:
    print(f'Skipped (endpoint not deployed): {str(e)[:200]}')


In [ ]:
# Trigger the first refresh to compute initial statistics
try:
    refresh = w.quality_monitors.run_refresh(table_name=inference_table)
    print(f"Refresh triggered. Refresh ID: {refresh.refresh_id}")
    print(f"State  : {refresh.state}")
    print()
    print("The monitor dashboard will appear in the Catalog Explorer:")
    print(f"  Catalog > {CATALOG} > {SCHEMA} > {inference_table.split('.')[-1]}")
    print("  Click the 'Quality' tab to view the Lakehouse Monitor dashboard.")
    print()
    print("The dashboard shows:")
    print("  - Data profile (distributions of request/response fields)")
    print("  - Drift metrics (Jensen-Shannon divergence vs. baseline window)")
    print("  - Data quality (null rates, schema violations)")
    print("  - Custom metrics (if configured via monitor_metric_config)")
except Exception as e:
    print(f"Refresh note: {e}")
    print("Re-run this cell once the monitor is successfully created.")

## D. Correlation Analysis — The Final Insight

This is the payoff moment. The whole lab series was building toward one question:

> **Is there a pattern between how clearly a company explains itself in its S-1 and how the stock performs in the first 12 months?**

We join `stock_performance` (12-month returns) with `clarity_scores` (automated clarity ratings of each S-1). The analyst hypothesis: companies that communicate clearly — concrete numbers, plain language, specific risk factors — may also be the ones with disciplined management, which in turn may correlate with better post-IPO performance.

We inspect the full distribution, then isolate the extremes: top 5 vs. bottom 5. The clarity scores in `score_json` contain both a numeric score and a justification — enough to reason qualitatively about each company.

In [ ]:
# Full distribution: all companies sorted by return
print("=== FULL DISTRIBUTION: Clarity Scores vs. 12-Month Returns ===")
display(spark.sql(f"""
    SELECT
        s.company,
        s.ticker,
        s.twelve_month_return_pct,
        c.score_json
    FROM {CATALOG}.{SCHEMA}.stock_performance s
    JOIN {CATALOG}.{SCHEMA}.clarity_scores c ON UPPER(s.ticker) = UPPER(c.ticker)
    ORDER BY s.twelve_month_return_pct DESC
"""))

In [ ]:
# The moment of truth
print("=== TOP 5 PERFORMERS — Clarity Scores ===")
display(spark.sql(f"""
    SELECT s.company, s.ticker, s.twelve_month_return_pct, c.score_json
    FROM {CATALOG}.{SCHEMA}.stock_performance s
    JOIN {CATALOG}.{SCHEMA}.clarity_scores c ON UPPER(s.ticker) = UPPER(c.ticker)
    ORDER BY s.twelve_month_return_pct DESC
    LIMIT 5
"""))

print("\n=== BOTTOM 5 PERFORMERS — Clarity Scores ===")
display(spark.sql(f"""
    SELECT s.company, s.ticker, s.twelve_month_return_pct, c.score_json
    FROM {CATALOG}.{SCHEMA}.stock_performance s
    JOIN {CATALOG}.{SCHEMA}.clarity_scores c ON UPPER(s.ticker) = UPPER(c.ticker)
    ORDER BY s.twelve_month_return_pct ASC
    LIMIT 5
"""))

print("\nIs there a pattern? Compare the clarity scores of top vs bottom performers.")
print("This is the question the firm's analysts wanted answered — and now there's data to discuss.")

In [ ]:
# Optional: calculate correlation coefficient
# Extract numeric score from score_json and compute Pearson correlation with return
try:
    corr_df = spark.sql(f"""
        SELECT
            s.ticker,
            s.twelve_month_return_pct,
            CAST(
                get_json_object(c.score_json, '$.score') AS DOUBLE
            ) AS clarity_score
        FROM {CATALOG}.{SCHEMA}.stock_performance s
        JOIN {CATALOG}.{SCHEMA}.clarity_scores c ON UPPER(s.ticker) = UPPER(c.ticker)
        WHERE get_json_object(c.score_json, '$.score') IS NOT NULL
    """)

    # Pearson correlation via Spark
    from pyspark.sql.functions import corr as spark_corr

    pearson = corr_df.select(
        spark_corr("clarity_score", "twelve_month_return_pct").alias("pearson_r")
    ).first()["pearson_r"]

    if pearson is not None:
        direction = "positive" if pearson > 0 else "negative"
        strength  = "strong" if abs(pearson) > 0.5 else ("moderate" if abs(pearson) > 0.3 else "weak")
        print(f"Pearson correlation (clarity score vs. 12-month return): r = {pearson:.3f}")
        print(f"Interpretation: {strength} {direction} correlation")
        print()
        if abs(pearson) > 0.3:
            print("There IS a statistically meaningful relationship worth investigating further.")
        else:
            print("The correlation is weak — clarity alone may not predict returns.")
            print("Other factors (market conditions, sector, valuation) likely dominate.")
    else:
        print("Could not compute correlation — check that score_json contains a 'score' field.")
except Exception as e:
    print(f"Correlation computation note: {e}")
    print("Verify that clarity_scores.score_json is a valid JSON string with a 'score' key.")

## E. The Feedback Loop

Monitoring is only useful if it drives action. The production workflow is a closed loop:

```
Production Endpoint → Inference Table → Lakehouse Monitor → Alert
    ↑                                                        ↓
    └── Redeploy ← Improve Rubric ← Re-evaluate (Lab 06) ←──┘
```

**Each stage:**

| Stage | Tool | Action |
|---|---|---|
| **Production Endpoint** | Model Serving (Lab 07) | Serves requests, logs to inference table |
| **Inference Table** | `auto_capture_config` (this lab) | Delta table of all requests + responses |
| **Lakehouse Monitor** | `quality_monitors.create()` (this lab) | Detects drift, computes quality metrics |
| **Alert** | Databricks Alerts or Workflows | Notifies the team when drift exceeds threshold |
| **Re-evaluate** | `mlflow.evaluate` (Lab 06) | Measure whether quality degraded |
| **Improve Rubric** | Prompt engineering (Lab 03) | Update the clarity scoring prompt or system prompt |
| **Redeploy** | `update_config_and_wait` (Lab 07) | Push new version with zero downtime |

**What triggers the loop?**

- **Drift alert:** Jensen-Shannon divergence on the response distribution exceeds a threshold — responses are getting shorter, or refusal rate is climbing, suggesting prompt regression or upstream data change.
- **Quality alert:** Latency P95 or error rate spikes.
- **Business alert:** The correlation insight changes — a quarterly re-run of the clarity vs. returns analysis shows the relationship weakening, suggesting the rubric needs recalibration for newer IPO cohorts.



## Before / After

**Before this lab:**
- Endpoint is live but completely blind — no visibility into what queries it receives or how it responds
- No alerting if quality degrades
- The correlation question has data (Lab 06) but no systematic analysis
- No feedback mechanism to improve the system over time

**After this lab:**
- Every request/response is logged to a Delta inference table via `auto_capture_config`
- Lakehouse Monitor runs automated drift detection (Jensen-Shannon divergence) daily
- Top 5 vs. bottom 5 correlation analysis is visible — the analyst question is answered
- The full feedback loop is understood: Monitor → Alert → Re-evaluate → Improve → Redeploy
- The product is complete: data pipeline → agent → scoring → evaluation → deployment → monitoring

## Final Scorecard

At this point all features should show YES. Run the scorecard to confirm the full system is intact end-to-end.

In [ ]:
# Return key results via notebook exit (enables API result inspection)
import json as _json
_results = {}
try:
    _results["lab"] = "08-monitoring"
    _results["status"] = "completed"
except Exception as e:
    _results["error"] = str(e)[:300]
dbutils.notebook.exit(_json.dumps(_results))


In [ ]:
# Scorecard -- tracks cumulative progress
try:
    import sys
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _parent = "/Workspace" + "/".join(_nb_path.split("/")[:-1])
    if _parent not in sys.path:
        sys.path.insert(0, _parent)
    from shared.lab_utils import get_scorecard
    get_scorecard()
except Exception as e:
    print(f"Scorecard skipped: {e}")

In [ ]:
# Uncomment to clean up resources
# w.serving_endpoints.delete("ipo-analyzer-endpoint")
# vsc.delete_endpoint("ipo_analyzer_vs_endpoint")
# print("Resources deleted. Lab complete!")